In [1]:
#| include: false
# Disable tqdm progress bars; this is a hidden cell on the docs page
import os

os.environ["TQDM_DISABLE"] = "1"

# Find which brain region a coordinate is in

`get_labels` takes coordinates and returns the Allen region id at each one. Coordinates are in metres relative to bregma, ordered ML, AP, DV, with DV negative below bregma; see [Coordinates and volumes](../explanation/coordinates-and-volumes.qmd).

In [2]:
import numpy as np
import pandas as pd

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
xyz = np.array([[-0.002, -0.002, -0.0015],  # ML, AP, DV in metres from bregma
                [ 0.002, -0.002, -0.0015]])
ids = ba.get_labels(xyz)
ba.regions.id2acronym(ids)

array(['CA1', 'CA1'], dtype=object)

The input is an (n, 3) array. A single point can also be given as an array of three values, and returns a single id:

In [3]:
print(ba.get_labels(np.array([-0.002, -0.002, -0.0015])))

382


## Get names as well as acronyms

`ba.regions.get` returns the acronym, full name, colour and level of each id.

In [4]:
regions = ba.regions.get(ids)
pd.DataFrame({'id': regions['id'], 'acronym': regions['acronym'],
              'name': regions['name'], 'level': regions['level']})

,id,acronym,name,level
0,382,CA1,Field CA1,8
1,382,CA1,Field CA1,8


## Get a coarser region

The `mapping` argument returns the region at a coarser level of the hierarchy. CA1 is a region of its own in the Beryl mapping, and part of HPF, the hippocampal formation, in Cosmos. See [Brain region mappings](../explanation/brain-region-mappings.qmd) for the levels each mapping keeps.

In [5]:
for mapping in ('Beryl', 'Cosmos'):
    print(mapping, ba.regions.id2acronym(ba.get_labels(xyz, mapping=mapping)))

Beryl ['CA1' 'CA1']
Cosmos ['HPF' 'HPF']


## Tell left from right

With a lateralised mapping such as `'Allen-lr'`, regions in the left hemisphere have negative ids. `id2acronym` returns the same acronym for both; see [Tell left from right hemisphere](tell-left-from-right-hemisphere.ipynb).

In [6]:
ids_lr = ba.get_labels(xyz, mapping='Allen-lr')
print(ids_lr, ba.regions.id2acronym(ids_lr))

[-382  382] ['CA1' 'CA1']


## Points outside the brain

| Point | Result |
|---|---|
| Inside the atlas volume, outside the brain | id 0, `void` |
| Outside the atlas volume | `ValueError` |
| Outside the atlas volume, with `mode='clip'` | The region of the nearest voxel on the edge of the volume, `void` in the example below |
| A coordinate containing NaN | id 0, `void`, the same id as a point outside the brain |

In [7]:
above_brain = np.array([[0, -0.002, 0.0002]])  # above the brain
outside_volume = np.array([[0, 0, 0.05]])
missing = np.array([[np.nan, -0.002, -0.0015]])

print(ba.get_labels(above_brain))
try:
    ba.get_labels(outside_volume)
except ValueError as err:
    print('ValueError:', err)
print(ba.regions.id2acronym(ba.get_labels(outside_volume, mode='clip')))
print(ba.get_labels(missing))

0
ValueError: At least one z value lies outside of the atlas volume.
['void']
0


## Coordinates in micrometres or in the Allen CCF

`get_labels` takes metres, so divide micrometres from bregma by `1e6`. Allen CCF coordinates are measured from a corner of the atlas volume, not from bregma; convert them first, as in [Convert coordinates from the Allen CCF or a registration tool](bring-your-own-coordinates.ipynb).

In [8]:
xyz_um = np.array([[-2000, -2000, -1500]])  # micrometres from bregma
ba.regions.id2acronym(ba.get_labels(xyz_um / 1e6))

array(['CA1'], dtype=object)

## The atlas changes the answer

The scaled atlases stretch and squeeze the Allen CCF, so the same coordinate can fall in a different region. Use the atlas your coordinates were made in; see [Atlas spaces and transforms](../explanation/atlas-spaces.qmd).

In [9]:
from iblatlas.atlas import NeedlesAtlas

point = np.array([[-0.001, -0.0025, -0.003]])
print('AllenAtlas  ', ba.regions.id2acronym(ba.get_labels(point)))
needles = NeedlesAtlas()
print('NeedlesAtlas', needles.regions.id2acronym(needles.get_labels(point)))

AllenAtlas   ['APN']


NeedlesAtlas ['MB']


## Precision

The lookup returns the region of the nearest voxel, and voxels are 25 µm apart by default. Near a boundary the 10 µm atlas is finer. It is a larger download; see [Data downloads and caching](../explanation/data-downloads-and-caching.qmd).

```python
ba = AllenAtlas(res_um=10)
```

## Related

- [From coordinates to a brain figure](../tutorials/from-coordinates-to-a-brain-figure.ipynb): looking up many points and summarising them by region